This notebook implements the algorithm described in the following paper:

S. Boyd, "Multitone signals with low crest factor," in _IEEE Transactions on Circuits and Systems_, vol. 33, no. 10, pp. 1018-1022, Oct. 1986, doi: 10.1109/TCS.1986.1085837.

In [ ]:
import math
import numpy as np
from scipy.fft import fft, fftfreq
from scipy.integrate import trapezoid
import matplotlib.pyplot as plt
import seaborn as sns

import sys
if '..' not in sys.path:
    sys.path = ['..'] + sys.path
from multitone import *

#### Compute the time series and crest factors for the first 100 tones

In [ ]:
F0 = 10e-3     # [Hz]
ω0 = 2 * np.pi * F0 # [rad/s]
T = 1 / F0     # [s]
dt = T / 5000  # [s]
fs = 1 / dt    # [Hz]
N_T = 2
t = np.r_[0 : N_T * T + dt/2 : dt]
tend = t[-1]   # [s]
print(f'Simulation duration: {tend} s.')
N_tones = np.arange(1, 101)
U = {N: multitone(t, N, algorithm='boyd', phases='newman', w0=2 * np.pi * F0) for N in N_tones}
CF = {N: compute_crest_factor(u, 1 / fs) for N, u in U.items()}

Check that `multitone_signal` and `multitone` give the same results

In [ ]:
N = 50
A, ω, ϕ = compute_multitone_pars(N, algorithm='boyd', phases='newman', w0=ω0)
assert np.allclose(
    U[N],
    multitone_signal(t, A, ω, ϕ)
), 'multitone and multitone_signal do not give the same results'

#### Plot of the crest factor as a function of the number of tones

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(3.5,2.5))
ax.plot(N_tones, CF.values(), 'ko-', markerfacecolor='w', markersize=3)
ax.set_xlabel('No. of tones')
ax.set_ylabel('Crest factor')
sns.despine()
fig.tight_layout()
plt.savefig('CF_Boyd.pdf')

Make sure that the RMS obtained from the Fourier coefficients matches the value obtained from the time series:

In [ ]:
from scipy.integrate import trapezoid
u = U[N]
F = np.array([(i + 1) * F0 for i in range(N)])
C = compute_fourier_coeffs(F, t, u)
c0 = trapezoid(u, t) / (t[-1] - t[0])
rms = RMS(u, t)
M = np.max(np.abs(u))
CF = M / rms
assert np.allclose(rms, RMS_from_complex_fourier_coeffs(C, c0))
print('Max value: {:g}'.format(M))
print('RMS value: {:g}'.format(rms))
print('Crest factor for N = {} tones: {:g} ({:g} dB)'.format(N, CF, 20 * np.log10(CF)))

Plot the time series, its distribution and Fourier spectra for a number of tones equal to 32.

In [ ]:
n, x = compute_amplitude_distribution(u, bins='fd')

uf = fft(u)
ups = 2 / u.size * np.abs(uf[:u.size // 2])
freqs = fftfreq(u.size, 1 / fs)[:u.size//2]
coeffs = compute_fourier_coeffs(F, t, u, ϕ, A)

with_Fu = False
n_rows = 2 + with_Fu
if with_Fu:
    w, h = 6, 5
else:
    w, h = 3.5, 2.5
fig,ax = plt.subplots(n_rows, 1, figsize=(w, h))
ax[0].plot(t, u, lw=0.75)
yl = ax[0].get_ylim()
ax[0].vlines(np.arange(N_T + 1) * T, yl[0], yl[1], color='tab:red', lw=0.75, ls=':')
ax[0].set_xlabel('Time [s]')
ax[0].set_ylabel('x')

if with_Fu:
    ax[1].plot(x, n, 'k', lw=0.75)
    ax[1].set_xlim([0, 2])
    ax[1].set_ylim([0, 1])
    ax[1].set_xlabel('Amplitude')
    ax[1].set_ylabel(r'$F_u(a)$')
    axx = ax[2]
else:
    axx = ax[1]
    
coeff = 1 if F0 > 0.1 else 1e3

idx = ups > 1/N
axx.vlines(freqs[idx] * coeff, 0 * ups[idx], ups[idx] * 0 + 2 / N, lw=0.5)
axx.plot(F * coeff, np.abs(coeffs), 'o', color='tab:red', markerfacecolor='tab:red', markersize=2)
axx.set_xlim([0, (N + 2) * F0 * coeff])
axx.set_ylim([0, 2 / N * 1.1])
axx.set_yticks([0, 2 / N])
axx.set_xlabel('Frequency [{}Hz]'.format('m' if coeff == 1e3 else ''))
axx.set_ylabel('|FFT(x)|')

sns.despine()
fig.tight_layout()
plt.savefig('figures/multitone_boyd.pdf')